# 00 — Data Inspection (LendingClub Accepted Loans)

Goal: validate the dataset, understand `loan_status`, and define a clean modelling label:
- `default_flag = 1` for **Charged Off** (and **Default** if present)
- `default_flag = 0` for **Fully Paid**
All other statuses (e.g. Current, Late, Grace Period) are excluded from modelling because outcomes are not final.

In [1]:
import pandas as pd

CSV_PATH = "data/raw/accepted_2007_to_2018Q4.csv"

In [2]:
df = pd.read_csv(CSV_PATH, low_memory=False)
df.shape

(2260701, 151)

In [3]:
len(df.columns), df.columns.tolist()[:40]

(151,
 ['id',
  'member_id',
  'loan_amnt',
  'funded_amnt',
  'funded_amnt_inv',
  'term',
  'int_rate',
  'installment',
  'grade',
  'sub_grade',
  'emp_title',
  'emp_length',
  'home_ownership',
  'annual_inc',
  'verification_status',
  'issue_d',
  'loan_status',
  'pymnt_plan',
  'url',
  'desc',
  'purpose',
  'title',
  'zip_code',
  'addr_state',
  'dti',
  'delinq_2yrs',
  'earliest_cr_line',
  'fico_range_low',
  'fico_range_high',
  'inq_last_6mths',
  'mths_since_last_delinq',
  'mths_since_last_record',
  'open_acc',
  'pub_rec',
  'revol_bal',
  'revol_util',
  'total_acc',
  'initial_list_status',
  'out_prncp',
  'out_prncp_inv'])

In [4]:
df["loan_status"].value_counts(dropna=False).head(20)

loan_status
Fully Paid                                             1076751
Current                                                 878317
Charged Off                                             268559
Late (31-120 days)                                       21467
In Grace Period                                           8436
Late (16-30 days)                                         4349
Does not meet the credit policy. Status:Fully Paid        1988
Does not meet the credit policy. Status:Charged Off        761
Default                                                     40
NaN                                                         33
Name: count, dtype: int64

In [5]:
FINAL_STATUSES = ["Fully Paid", "Charged Off", "Default"]

df_final = df[df["loan_status"].isin(FINAL_STATUSES)].copy()

df_final["default_flag"] = df_final["loan_status"].isin(["Charged Off", "Default"]).astype(int)

df_final["default_flag"].value_counts(), df_final.shape

(default_flag
 0    1076751
 1     268599
 Name: count, dtype: int64,
 (1345350, 152))

## Class balance
keep only final outcomes for modelling to avoid label noise from incomplete loans.
Below is the default rate in the final-outcome subset.

In [6]:
default_rate = df_final["default_flag"].mean()
n = len(df_final)
n_default = int(df_final["default_flag"].sum())

{
    "n_final_outcomes": n,
    "n_default": n_default,
    "default_rate_pct": round(100 * default_rate, 2)
}

{'n_final_outcomes': 1345350,
 'n_default': 268599,
 'default_rate_pct': np.float64(19.96)}